# Map cells across data

In [1]:
import os
os.chdir('/nfs/users/nfs_v/vm11/projects/hdca_v2_reprocessing/03_link_reprocessed_to_author')

In [9]:
from datetime import date
import pandas as pd
import numpy as np
import sys
sys.path.append("..")
import utils.fetch_dbs
#import utils.bcutils
import json
import mysql.connector
from pathlib import Path
import scanpy as sc

In [20]:
import logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s : %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger(__name__)


In [5]:
storage_path = '/lustre/scratch124/cellgen/haniffa/users/ar32/fresh_public_object_curations/output_files/'
publication_id = 'vento_tormo_2018_placenta_decidua'
file_path = Path(storage_path) / publication_id / f'{publication_id}_scRNA_main.h5ad'

# example author object

In [21]:
logger.info(f"Reading data from {file_path}")
adata = sc.read_h5ad(file_path, backed = 'r')


11:23:43 INFO __main__: Reading data from /lustre/scratch124/cellgen/haniffa/users/ar32/fresh_public_object_curations/output_files/vento_tormo_2018_placenta_decidua/vento_tormo_2018_placenta_decidua_scRNA_main.h5ad


In [11]:
adata

AnnData object with n_obs × n_vars = 64734 × 31764 backed at '/lustre/scratch124/cellgen/haniffa/users/ar32/fresh_public_object_curations/output_files/vento_tormo_2018_placenta_decidua/vento_tormo_2018_placenta_decidua_scRNA_main.h5ad'
    obs: 'Fetus', 'location', 'final_cluster', 'annotation', 'barcode', 'obs_unit_id', 'original_author_annotation_broad', 'original_author_annotation_fine'
    var: 'orig_index', 'hgnc'

In [13]:
logger.info(f"Reading data from {file_path}")
adata.obs['author_obs_names'] = adata.obs.index # Retain original cell-barcode names
adata.obs['author_obs_names'] = adata.obs['author_obs_names'].astype(str) # Ensure they are strings

In [10]:
adata.obs['obs_unit_id'].unique()

['FCA7167219', 'FCA7167221', 'FCA7167222', 'FCA7167223', 'FCA7167224', ..., 'FCA7474065', 'FCA7474068', 'FCA7511881', 'FCA7511882', 'FCA7511884']
Length: 25
Categories (25, object): ['FCA7167219', 'FCA7167221', 'FCA7167222', 'FCA7167223', ..., 'FCA7474068', 'FCA7511881', 'FCA7511882', 'FCA7511884']


✓ Cell executed at 2026-06-03 10:54:23


In [11]:
# EMAT tab

# load sdrf
sdrf = pd.read_csv('https://ftp.ebi.ac.uk/biostudies/fire/E-MTAB-/701/E-MTAB-6701/Files/E-MTAB-6701.sdrf.txt', sep='\t')
sdrf.head()


,Source Name,Comment[ENA_SAMPLE],Comment[BioSD_SAMPLE],Characteristics[organism],Characteristics[individual],Characteristics[sex],Characteristics[developmental stage],Characteristics[organism part],Characteristics[disease],Characteristics[clinical information],Characteristics[FACS marker],Characteristics[run],Comment[single cell isolation],Comment[library construction],Comment[input molecule],Comment[primer],Comment[end bias],Material Type,Protocol REF,Protocol REF.1,Protocol REF.2,Extract Name,Comment[LIBRARY_LAYOUT],Comment[LIBRARY_SELECTION],Comment[LIBRARY_SOURCE],Comment[LIBRARY_STRAND],Comment[LIBRARY_STRATEGY],Comment[NOMINAL_LENGTH],Comment[NOMINAL_SDEV],Protocol REF.3,Performer,Assay Name,Technology Type,Comment[ENA_EXPERIMENT],Scan Name,Comment[ENA_RUN],Comment[BAM_URI],Comment[FASTQ_URI],Comment[FASTQ_URI].1,Comment[FASTQ_URI].2,Comment[SPOT_LENGTH],Comment[READ_INDEX_1_BASE_COORD],Protocol REF.4,Derived Array Data File,Comment [Derived ArrayExpress FTP file],Protocol REF.5,Derived Array Data File.1,Comment [Derived ArrayExpress FTP file].1,Comment[umi barcode read],Comment[umi barcode offset],Comment[umi barcode size],Comment[cell barcode read],Comment[cell barcode offset],Comment[cell barcode size],Comment[cDNA read],Comment[cDNA read offset],Comment[cDNA read size],Comment[sample barcode read],Comment[sample barcode offset],Comment[sample barcode size],Comment[read1 file],Comment[read2 file],Comment[index1 file],Factor Value[organism part],Factor Value[FACS marker]
0,FCA7167219,ERS2657608,SAMEA4837741,Homo sapiens,D6,female,adult,decidua,normal,6 to 12 weeks gestation,CD45+,FCA7167219,10x,10xV2,polyA RNA,oligo-dT,3 prime tag,cell,P-MTAB-73444,P-MTAB-73445,P-MTAB-73446,FCA7167219,PAIRED,Oligo-dT,TRANSCRIPTOMIC SINGLE CELL,not applicable,RNA-Seq,666,5,P-MTAB-73447,Sanger institute,FCA7167219,sequencing assay,ERX2756708,FCA7167219.bam,ERR2743635,ftp://ftp.sra.ebi.ac.uk/vol1/run/ERR274/ERR274...,ftp://ftp.ebi.ac.uk/pub/databases/microarray/d...,ftp://ftp.ebi.ac.uk/pub/databases/microarray/d...,ftp://ftp.ebi.ac.uk/pub/databases/microarray/d...,946,474,P-MTAB-73448,raw_data_10x.txt,ftp://ftp.ebi.ac.uk/pub/databases/microarray/d...,P-MTAB-73448,meta_10x.txt,ftp://ftp.ebi.ac.uk/pub/databases/microarray/d...,read1,16,10,read1,0,16,read2,0,98,index1,0,8,FCA7167219_R1.fq.gz,FCA7167219_R2.fq.gz,FCA7167219_I1.fq.gz,decidua,CD45+
1,FCA7167221,ERS2657610,SAMEA4837743,Homo sapiens,D7,female,adult,decidua,normal,6 to 12 weeks gestation,CD45+,FCA7167221,10x,10xV2,polyA RNA,oligo-dT,3 prime tag,cell,P-MTAB-73444,P-MTAB-73445,P-MTAB-73446,FCA7167221,PAIRED,Oligo-dT,TRANSCRIPTOMIC SINGLE CELL,not applicable,RNA-Seq,666,5,P-MTAB-73447,Sanger institute,FCA7167221,sequencing assay,ERX2756710,FCA7167221.bam,ERR2743637,ftp://ftp.sra.ebi.ac.uk/vol1/run/ERR274/ERR274...,ftp://ftp.ebi.ac.uk/pub/databases/microarray/d...,ftp://ftp.ebi.ac.uk/pub/databases/microarray/d...,ftp://ftp.ebi.ac.uk/pub/databases/microarray/d...,946,474,P-MTAB-73448,raw_data_10x.txt,ftp://ftp.ebi.ac.uk/pub/databases/microarray/d...,P-MTAB-73448,meta_10x.txt,ftp://ftp.ebi.ac.uk/pub/databases/microarray/d...,read1,16,10,read1,0,16,read2,0,98,index1,0,8,FCA7167221_R1.fq.gz,FCA7167221_R2.fq.gz,FCA7167221_I1.fq.gz,decidua,CD45+
2,FCA7167222,ERS2657611,SAMEA4837744,Homo sapiens,D7,female,adult,decidua,normal,6 to 12 weeks gestation,CD45-,FCA7167222,10x,10xV2,polyA RNA,oligo-dT,3 prime tag,cell,P-MTAB-73444,P-MTAB-73445,P-MTAB-73446,FCA7167222,PAIRED,Oligo-dT,TRANSCRIPTOMIC SINGLE CELL,not applicable,RNA-Seq,666,5,P-MTAB-73447,Sanger institute,FCA7167222,sequencing assay,ERX2756711,FCA7167222.bam,ERR2743638,ftp://ftp.sra.ebi.ac.uk/vol1/run/ERR274/ERR274...,ftp://ftp.ebi.ac.uk/pub/databases/microarray/d...,ftp://ftp.ebi.ac.uk/pub/databases/microarray/d...,ftp://ftp.ebi.ac.uk/pub/databases/microarray/d...,946,474,P-MTAB-73448,raw_data_10x.txt,ftp://ftp.ebi.ac.uk/pub/databases/microarray/d...,P-MTAB-73448,meta_10x.txt,ftp://ftp.ebi.ac.uk/pub/databases/microarray/d...,read1,16,10


✓ Cell executed at 2026-06-03 10:54:54


In [ ]:
adata.obs

In [12]:
# merge adata.obs with sdrf on library_id <-> Source Name, preserving cell-barcode index
merged = (
    adata.obs.reset_index()
    .merge(sdrf, how='left', left_on='obs_unit_id', right_on='Source Name')
    .set_index(adata.obs.index.name or 'index')
)
merged

,Fetus,location,final_cluster,annotation,barcode,obs_unit_id,original_author_annotation_broad,original_author_annotation_fine,author_obs_names,Source Name,Comment[ENA_SAMPLE],Comment[BioSD_SAMPLE],Characteristics[organism],Characteristics[individual],Characteristics[sex],Characteristics[developmental stage],Characteristics[organism part],Characteristics[disease],Characteristics[clinical information],Characteristics[FACS marker],Characteristics[run],Comment[single cell isolation],Comment[library construction],Comment[input molecule],Comment[primer],Comment[end bias],Material Type,Protocol REF,Protocol REF.1,Protocol REF.2,Extract Name,Comment[LIBRARY_LAYOUT],Comment[LIBRARY_SELECTION],Comment[LIBRARY_SOURCE],Comment[LIBRARY_STRAND],Comment[LIBRARY_STRATEGY],Comment[NOMINAL_LENGTH],Comment[NOMINAL_SDEV],Protocol REF.3,Performer,Assay Name,Technology Type,Comment[ENA_EXPERIMENT],Scan Name,Comment[ENA_RUN],Comment[BAM_URI],Comment[FASTQ_URI],Comment[FASTQ_URI].1,Comment[FASTQ_URI].2,Comment[SPOT_LENGTH],Comment[READ_INDEX_1_BASE_COORD],Protocol REF.4,Derived Array Data File,Comment [Derived ArrayExpress FTP file],Protocol REF.5,Derived Array Data File.1,Comment [Derived ArrayExpress FTP file].1,Comment[umi barcode read],Comment[umi barcode offset],Comment[umi barcode size],Comment[cell barcode read],Comment[cell barcode offset],Comment[cell barcode size],Comment[cDNA read],Comment[cDNA read offset],Comment[cDNA read size],Comment[sample barcode read],Comment[sample barcode offset],Comment[sample barcode size],Comment[read1 file],Comment[read2 file],Comment[index1 file],Factor Value[organism part],Factor Value[FACS marker]
index,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
FCA7167219_AAACGGGAGCTAGTTC,D6,Decidua,2,dNK2,AAACGGGAGCTAGTTC,FCA7167219,dNK2,dNK2,FCA7167219_AAACGGGAGCTAGTTC,FCA7167219,ERS2657608,SAMEA4837741,Homo sapiens,D6,female,adult,decidua,normal,6 to 12 weeks gestation,CD45+,FCA7167219,10x,10xV2,polyA RNA,oligo-dT,3 prime tag,cell,P-MTAB-73444,P-MTAB-73445,P-MTAB-73446,FCA7167219,PAIRED,Oligo-dT,TRANSCRIPTOMIC SINGLE CELL,not applicable,RNA-Seq,666,5,P-MTAB-73447,Sanger institute,FCA7167219,sequencing assay,ERX2756708,FCA7167219.bam,ERR2743635,ftp://ftp.sra.ebi.ac.uk/vol1/run/ERR274/ERR274...,ftp://ftp.ebi.ac.uk/pub/databases/microarray/d...,ftp://ftp.ebi.ac.uk/pub/databases/microarray/d...,ftp://ftp.ebi.ac.uk/pub/databases/microarray/d...,946,474,P-MTAB-73448,raw_data_10x.txt,ftp://ftp.ebi.ac.uk/pub/databases/microarray/d...,P-MTAB-73448,meta_10x.txt,ftp://ftp.ebi.ac.uk/pub/databases/microarray/d...,read1,16,10,read1,0,16,read2,0,98,index1,0,8,FCA7167219_R1.fq.gz,FCA7167219_R2.fq.gz,FCA7167219_I1.fq.gz,decidua,CD45+
FCA7167219_AAACGGGCATTGGCGC,D6,Decidua,5,dNK1,AAACGGGCATTGGCGC,FCA7167219,dNK1,dNK1,FCA7167219_AAACGGGCATTGGCGC,FCA7167219,ERS2657608,SAMEA4837741,Homo sapiens,D6,female,adult,decidua,normal,6 to 12 weeks gestation,CD45+,FCA7167219,10x,10xV2,polyA RNA,oligo-dT,3 prime tag,cell,P-MTAB-73444,P-MTAB-73445,P-MTAB-73446,FCA7167219,PAIRED,Oligo-dT,TRANSCRIPTOMIC SINGLE CELL,not applicable,RNA-Seq,666,5,P-MTAB-73447,Sanger institute,FCA7167219,sequencing assay,ERX2756708,FCA7167219.bam,ERR2743635,ftp://ftp.sra.ebi.ac.uk/vol1/run/ERR274/ERR274...,ftp://ftp.ebi.ac.uk/pub/databases/microarray/d...,ftp://ftp.ebi.ac.uk/pub/databases/microarray/d...,ftp://ftp.ebi.ac.uk/pub/databases/microarray/d...,946,474,P-MTAB-73448,raw_data_10x.txt,ftp://ftp.ebi.ac.uk/pub/databases/microarray/d...,P-MTAB-73448,meta_10x.txt,ftp://ftp.ebi.ac.uk/pub/databases/microarray/d...,read1,16,10,read1,0,16,read2,0,98,index1,0,8,FCA7167219_R1.fq.gz,FCA7167219_R2.fq.gz,FCA7167219_I1.fq.gz,decidua,CD45+
FCA7167219_AAACGGGTCGCGATCG,D6,Decidua,8,Tcells,AAACGGGTCGCGATCG,FCA7167219,Tcells,Tcells,FCA7167219_AAACGGGTCGCGATCG,FCA7167219,ERS2657608,SAMEA4837741,Homo sapiens,D6,female,adult,decidua,normal,6 to 12 weeks gestation,CD45+,FCA7167219,10x,10xV2,polyA RNA,oligo-dT,3 prime tag,cell,P-MTAB-73444,P-MTAB-73445,P-MTAB-7


✓ Cell executed at 2026-06-03 10:55:02


In [ ]:
logger.warning("Only proceed if the indices match")

if not adata.obs.index.equals(merged.index):
    logger.error("Index mismatch between adata.obs and merged.index")
else:
    logger.info("Index matches between adata.obs and merged.index")
    adata.obs = merged


2026-06-03 10:55:12.941 | WARNING  | __main__:<module>:1 - Only process if the indices match
2026-06-03 10:55:12.945 | INFO     | __main__:<module>:6 - Index matches between adata.obs and merged.index



✓ Cell executed at 2026-06-03 10:55:12


In [14]:
adata

AnnData object with n_obs × n_vars = 64734 × 31764 backed at '/lustre/scratch124/cellgen/haniffa/users/ar32/fresh_public_object_curations/output_files/vento_tormo_2018_placenta_decidua/vento_tormo_2018_placenta_decidua_scRNA_main.h5ad'
    obs: 'Fetus', 'location', 'final_cluster', 'annotation', 'barcode', 'obs_unit_id', 'original_author_annotation_broad', 'original_author_annotation_fine', 'author_obs_names', 'Source Name', 'Comment[ENA_SAMPLE]', 'Comment[BioSD_SAMPLE]', 'Characteristics[organism]', 'Characteristics[individual]', 'Characteristics[sex]', 'Characteristics[developmental stage]', 'Characteristics[organism part]', 'Characteristics[disease]', 'Characteristics[clinical information]', 'Characteristics[FACS marker]', 'Characteristics[run]', 'Comment[single cell isolation]', 'Comment[library construction]', 'Comment[input molecule]', 'Comment[primer]', 'Comment[end bias]', 'Material Type', 'Protocol REF', 'Protocol REF.1', 'Protocol REF.2', 'Extract Name', 'Comment[LIBRARY_LAYO


✓ Cell executed at 2026-06-03 10:55:16


In [15]:
adata.obs.index

Index(['FCA7167219_AAACGGGAGCTAGTTC', 'FCA7167219_AAACGGGCATTGGCGC',
       'FCA7167219_AAACGGGTCGCGATCG', 'FCA7167219_AAAGATGAGCAATATG',
       'FCA7167219_AAAGATGAGTTCGCGC', 'FCA7167219_AAAGATGCATGTCGAT',
       'FCA7167219_AAAGATGGTCTCGTTC', 'FCA7167219_AAATGCCTCCCTTGCA',
       'FCA7167219_AACCATGCAGCAGTTT', 'FCA7167219_AACCGCGCATAGGATA',
       ...
       'FCA7511884_TTTCCTCTCTGGAGCC', 'FCA7511884_TTTGCGCAGCTAAACA',
       'FCA7511884_TTTGCGCGTGAAATCA', 'FCA7511884_TTTGCGCTCCCTAACC',
       'FCA7511884_TTTGGTTGTCCAGTGC', 'FCA7511884_TTTGGTTGTTAAAGAC',
       'FCA7511884_TTTGGTTTCCATGCTC', 'FCA7511884_TTTGTCACACCGAAAG',
       'FCA7511884_TTTGTCATCCGAGCCA', 'FCA7511884_TTTGTCATCTTGAGGT'],
      dtype='object', name='index', length=64734)


✓ Cell executed at 2026-06-03 10:55:28


In [ ]:
adata.obs.index = list(adata.obs.index)


✓ Cell executed at 2026-06-02 15:44:55


In [38]:
adata.obs['library_id'] = adata.obs['Comment[ENA_SAMPLE]']


✓ Cell executed at 2026-06-02 15:55:58


In [43]:
np.savetxt('library_ids.txt', adata.obs['library_id'].unique(), fmt='%s')


✓ Cell executed at 2026-06-02 15:58:04


FATAL:   While checking container encryption: could not open image /usr/bin/ils: failed to retrieve path for /usr/bin/ils: lstat /usr/bin/ils: no such file or directory

✓ Cell executed at 2026-06-02 15:56:27


In [ ]:
|